# 01: Preprocessing, Adult Income Dataset

Transforms the raw UCI Adult census file (Kohavi, 1996) into the analysis
dataset specified in Ch. 3, binary target (income >$50K = 1, base rate ~24%),
and explicit binary fairness attribute `sex_female` (female = protected
subgroup per Ch. 3).

Raw input: s3://osilesi-dissertation-data/raw/adult.data (never modified).
Output: s3://osilesi-dissertation-data/processed/adult_final_pruned.csv

In [13]:
import pandas as pd
import numpy as np

BUCKET = "osilesi-dissertation-data-2026"   # bucket name
pd.set_option("display.max_columns", None)

In [14]:
cols = ["age", "workclass", "fnlwgt", "education", "education_num",
        "marital_status", "occupation", "relationship", "race", "sex",
        "capital_gain", "capital_loss", "hours_per_week",
        "native_country", "income"]

adult = pd.read_csv(f"s3://{BUCKET}/raw/adult.data",
                    header=None, names=cols, skipinitialspace=True)

print(adult.shape)          # expect (32561, 15)
adult.head()

(32561, 15)


,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


In [15]:
for c in adult.columns:
    n = (adult[c] == "?").sum()
    if n: print(c, n)
# expect: workclass ~1836, occupation ~1843, native_country ~583

workclass 1836
occupation 1843
native_country 583


## Missing value strategy

Adult encodes missingness as "?" (workclass ~1,836; occupation ~1,843;
native_country ~583). Rows are NOT dropped: "?" is recoded to an explicit "Unknown" category,
which survives one-hot encoding as its own indicator column.

In [16]:
adult = adult.replace("?", "Unknown")

In [17]:
adult["target"] = (adult["income"] == ">50K").astype(int)
adult = adult.drop(columns=["income"])
print(adult["target"].value_counts(normalize=True))   # expect ~0.24 positive

target
0    0.75919
1    0.24081
Name: proportion, dtype: float64


## Drop decisions, Adult

- fnlwgt: census sampling weight, not a person-level attribute
- education: fully redundant with ordinal education_num
- native_country: sparse categorical
- relationship: proxy attribute
- sex (raw): replaced by explicit binary sex_female before encoding

In [18]:
drop_cols = ["fnlwgt", "education", "native_country", "relationship"]
adult = adult.drop(columns=drop_cols)
print(adult.shape)

(32561, 11)


## Fairness attribute construction

`sex_female` is built as an explicit binary column BEFORE one-hot
encoding so it can never be silently lost or renamed. It is the sole protected attribute in the design and is
protected from removal throughout the pipeline.

In [19]:
adult["sex_female"] = (adult["sex"] == "Female").astype(int)
adult = adult.drop(columns=["sex"])
print(adult["sex_female"].value_counts(normalize=True))   # ~0.33 female

sex_female
0    0.669205
1    0.330795
Name: proportion, dtype: float64


In [20]:
cat_cols = adult.select_dtypes(include="object").columns.tolist()
print("Encoding:", cat_cols)   # workclass, marital_status, occupation, race

adult = pd.get_dummies(adult, columns=cat_cols, dtype=int)
print(adult.shape)

Encoding: ['workclass', 'marital_status', 'occupation', 'race']
(32561, 43)


In [21]:
n_features = adult.shape[1] - 1        # exclude target
assert adult.shape[0] == 32561, f"Row count off: {adult.shape[0]}"
assert adult.isna().sum().sum() == 0,  "Missing values present"
assert "sex_female" in adult.columns,  "Fairness attribute lost"
print(f"Features: {n_features} (Chapter Three states 42)")

Features: 42 (Chapter Three states 42)


## Validation gate: PASSED [2026-08-01]

32,561 rows; 42 features (excl. target); 0 missing values; sex_female
present (~33% female); positive class ~24%. Output written to /processed.

In [11]:
adult.to_csv(f"s3://{BUCKET}/processed/adult_final_pruned.csv", index=False)